# Döntési fa — Tanítás

**Felelős:** Kovács Kornél

Ez a notebook tanít, kiértékel, és minden köztes adatot kiment Drive-ra. **Plot egy se** — a vizualizáció a `03_decision_tree_visualize.ipynb`-ben él, ami ezeket a mentett artefaktokat olvassa be.

A 2. szekcióban beállítható `VERSION` változó (`'v0' | 'vKNN' | 'v1'`) — minden mentett fájl és a `metrics.csv`-be kerülő modellnév is ezt a suffixet hordozza, így több verzió eredménye békésen megfér Drive-on.

Mentett kimenetek (Drive: `MODELS_DIR`, `{VERSION}` placeholder):
- `decision_tree_tuned_{VERSION}.joblib` — a hangolt modell
- `decision_tree_predictions_test_{VERSION}.npy` — a `best_model.predict(X_test)` eredménye
- `decision_tree_validation_curve_max_depth_{VERSION}.npz` — validation curve arrays
- `decision_tree_learning_curve_{VERSION}.npz` — learning curve arrays

Plus `results/metrics.csv` (Drive) — append-elt metrika sorok, modellnévben a `(VERSION)`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/03_decision_tree.ipynb)

## 1. Setup (Colab + lokális kompatibilis)

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git pull
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, validation_curve, learning_curve
import joblib

from src.config import RANDOM_SEED, CV_FOLDS, MODELS_DIR, ensure_dirs
from src.data_io import load_v1_data
from src.evaluation import append_metrics, evaluate_model

ensure_dirs()
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Adatbetöltés

In [ ]:
VERSION = 'v0'  # 'v0' (200k sample) | 'vKNN' (500k) | 'v1' (teljes 9.6M)

X_train, X_test, y_train, y_test, feature_names = load_v1_data(version=VERSION)

print(f'Verzió: {VERSION}')
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Target — train: mean={y_train.mean():.2f}, std={y_train.std():.2f}, '
      f'min={y_train.min()}, max={y_train.max()}')

## 3. Baseline: alap döntési fa

Default `DecisionTreeRegressor` paraméterhangolás nélkül — referenciapont a tuned változathoz.

In [ ]:
dt_default = DecisionTreeRegressor(random_state=RANDOM_SEED)

result_default, _ = evaluate_model(
    dt_default,
    X_train, y_train, X_test, y_test,
    model_name=f'Decision Tree (default, {VERSION})',
    notes=f'no tuning, version={VERSION}',
)

print(f'MAE: {result_default.mae:.3f}')
print(f'RMSE: {result_default.rmse:.3f}')
print(f'R²: {result_default.r2:.3f}')
print(f'Train idő: {result_default.train_time_sec:.2f} s')

append_metrics(result_default)

## 4. GridSearchCV hangolás (sample-en) + refit a teljes train-en

1. `TUNE_SAMPLE_SIZE` random sample a train-ből → ezen GridSearchCV
2. A legjobb paraméterekkel refit a teljes train-en
3. Mentés: modell + test predikciók + metrika

In [ ]:
TUNE_SAMPLE_SIZE = 500_000

param_grid = {
    'max_depth': [8, 10, 12, 15],
    'min_samples_leaf': [10, 50, 100],
    'min_samples_split': [2],
}

n_tune = min(TUNE_SAMPLE_SIZE, len(X_train))
rng = np.random.default_rng(RANDOM_SEED)
tune_idx = rng.choice(len(X_train), size=n_tune, replace=False)
X_tune, y_tune = X_train[tune_idx], y_train[tune_idx]

grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=RANDOM_SEED),
    param_grid=param_grid,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_tune, y_tune)

print(f'\nTune sample size: {n_tune}')
print(f'Legjobb paraméterek: {grid.best_params_}')
print(f'CV legjobb score (neg MAE): {grid.best_score_:.3f}')

best_model = DecisionTreeRegressor(**grid.best_params_, random_state=RANDOM_SEED)
result_tuned, y_pred_tuned = evaluate_model(
    best_model,
    X_train, y_train, X_test, y_test,
    model_name=f'Decision Tree (tuned, {VERSION})',
    notes=f'GridSearchCV cv={CV_FOLDS} on n={n_tune}, refit on full train ({VERSION})',
)

print(f'\nTeszt MAE: {result_tuned.mae:.3f}')
print(f'Teszt RMSE: {result_tuned.rmse:.3f}')
print(f'Teszt R²: {result_tuned.r2:.3f}')

append_metrics(result_tuned)

joblib.dump(best_model, MODELS_DIR / f'decision_tree_tuned_{VERSION}.joblib')
np.save(MODELS_DIR / f'decision_tree_predictions_test_{VERSION}.npy', y_pred_tuned)
print(f'\nMentve: {MODELS_DIR}/decision_tree_tuned_{VERSION}.joblib + decision_tree_predictions_test_{VERSION}.npy')

## 5. Validation curve (sample-en)

`max_depth` mentén — friss fitek egy 50k-os train sample-en. A pozitív MAE arrayokat mentjük .npz-be.

In [ ]:
DIAG_SAMPLE_SIZE = 50_000

diag_rng = np.random.default_rng(RANDOM_SEED)
diag_idx = diag_rng.choice(len(X_train), size=min(DIAG_SAMPLE_SIZE, len(X_train)), replace=False)
X_diag, y_diag = X_train[diag_idx], y_train[diag_idx]
print(f'Diagnosztikai sample: {X_diag.shape}')

vc_param_range = [3, 5, 7, 10, 15, 20, None]
vc_train_scores, vc_test_scores = validation_curve(
    DecisionTreeRegressor(random_state=RANDOM_SEED),
    X_diag, y_diag,
    param_name='max_depth',
    param_range=vc_param_range,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
vc_train_mae = -vc_train_scores
vc_test_mae = -vc_test_scores

vc_labels = np.array([str(p) if p is not None else 'None' for p in vc_param_range])
np.savez(
    MODELS_DIR / f'decision_tree_validation_curve_max_depth_{VERSION}.npz',
    param_range_labels=vc_labels,
    train_mae=vc_train_mae,
    test_mae=vc_test_mae,
)
print(f'Validation curve mentve: {MODELS_DIR}/decision_tree_validation_curve_max_depth_{VERSION}.npz')
for lbl, tr, te in zip(vc_labels, vc_train_mae.mean(axis=1), vc_test_mae.mean(axis=1)):
    print(f'  max_depth={lbl:>4s}: train MAE={tr:.3f}, CV MAE={te:.3f}')

## 6. Learning curve (sample-en)

A tuned hiperparaméterekkel, train_size mentén — ugyanaz a sample. .npz-be mentve.

In [ ]:
lc_train_sizes = np.linspace(0.1, 1.0, 8)
lc_sizes, lc_train_scores, lc_test_scores = learning_curve(
    DecisionTreeRegressor(**best_model.get_params()),
    X_diag, y_diag,
    train_sizes=lc_train_sizes,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    shuffle=True,
    random_state=RANDOM_SEED,
)
lc_train_mae = -lc_train_scores
lc_test_mae = -lc_test_scores

np.savez(
    MODELS_DIR / f'decision_tree_learning_curve_{VERSION}.npz',
    sizes=lc_sizes,
    train_mae=lc_train_mae,
    test_mae=lc_test_mae,
)
print(f'Learning curve mentve: {MODELS_DIR}/decision_tree_learning_curve_{VERSION}.npz')
for sz, tr, te in zip(lc_sizes, lc_train_mae.mean(axis=1), lc_test_mae.mean(axis=1)):
    print(f'  size={int(sz):>6d}: train MAE={tr:.3f}, CV MAE={te:.3f}')

## 7. Tanulságok

_A számokat itt érdemes összegezni; a vizualizációk a `03_decision_tree_visualize.ipynb`-ben._